In [1]:
import polars as pl
data_pl = pl.read_csv(r"../data/train.csv",encoding="shift_jis")


In [4]:
data_pl.head(1)

sample number,species number,樹種,含水率,9993.76781,9989.9107,9986.05359,9982.19648,9978.33937,9974.48227,9970.62516,9966.76805,9962.91094,9959.05383,9955.19672,9951.33962,9947.48251,9943.6254,9939.76829,9935.91118,9932.05407,9928.19697,9924.33986,9920.48275,9916.62564,9912.76853,9908.91142,9905.05432,9901.19721,9897.3401,9893.48299,9889.62588,9885.76877,9881.91166,9878.05456,9874.19745,9870.34034,…,4138.67729,4134.82018,4130.96307,4127.10596,4123.24886,4119.39175,4115.53464,4111.67753,4107.82042,4103.96331,4100.10621,4096.2491,4092.39199,4088.53488,4084.67777,4080.82066,4076.96356,4073.10645,4069.24934,4065.39223,4061.53512,4057.67801,4053.82091,4049.9638,4046.10669,4042.24958,4038.39247,4034.53536,4030.67826,4026.82115,4022.96404,4019.10693,4015.24982,4011.39271,4007.5356,4003.6785,3999.82139
i64,i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,1,"""イチョウ""",216.129032,0.41485,0.41465,0.41463,0.41476,0.41481,0.4147,0.41452,0.41427,0.41392,0.41364,0.41354,0.41355,0.41352,0.41337,0.41312,0.41284,0.4127,0.41265,0.41257,0.41242,0.4123,0.41226,0.41218,0.41193,0.41159,0.41129,0.41111,0.41102,0.41096,0.41098,0.41109,0.41116,0.41104,…,1.1985,1.19969,1.20339,1.20819,1.21138,1.21194,1.21262,1.21483,1.21592,1.21666,1.22043,1.22559,1.22777,1.22751,1.22862,1.23165,1.23629,1.24109,1.24147,1.23927,1.24074,1.24282,1.24082,1.23906,1.24128,1.24471,1.24783,1.25104,1.24925,1.24145,1.2362,1.23384,1.22981,1.22818,1.23087,1.23354,1.23219


In [18]:
data_pl.estimated_size()

16492099

In [5]:
data_new = data_pl.with_columns(
    pl.col("含水率").log().alias("含水率_log")
)

In [19]:
data_new.estimated_size()

16502675

In [21]:
exclude_cols = ["sample number", "species number", 
                "樹種", "含水率", "含水率_log"]
num_cols = [c for c in data_new.columns if c not in exclude_cols]

# 横方向の差分カラム名を作成
diff_cols = [f"{num_cols[i]}_diff_{num_cols[i+1]}"
             for i in range(len(num_cols)-1)]

# 横方向の差分を計算して新規カラムとして追加
diff_exprs = [
    (pl.col(num_cols[i+1]) - pl.col(num_cols[i])).alias(diff_cols[i])
    for i in range(len(num_cols)-1)
]

pl_diff_df = data_new.with_columns(diff_exprs)

In [38]:
pl_diff_df.estimated_size()

31.411913871765137

In [23]:
import polars as pl

# 数値列のみ対象（重要）
num_cols = [c for c, d in zip(pl_diff_df.columns, pl_diff_df.dtypes) if d in (pl.Float64, pl.Int64)]

# 分散チェックして残す列
valid_cols = [
    c for c in num_cols
    if pl_diff_df.select(pl.col(c).var()).item() != 0
]

# 元の非数値列も残す
other_cols = [c for c in pl_diff_df.columns if c not in num_cols]

# 結合
pl_diff_df_var = pl_diff_df.select(other_cols + valid_cols)

In [24]:
pl_diff_df_var.estimated_size()

31.411913871765137

In [31]:
pl_diff_df_var.head()

樹種,sample number,species number,含水率,9993.76781,9989.9107,9986.05359,9982.19648,9978.33937,9974.48227,9970.62516,9966.76805,9962.91094,9959.05383,9955.19672,9951.33962,9947.48251,9943.6254,9939.76829,9935.91118,9932.05407,9928.19697,9924.33986,9920.48275,9916.62564,9912.76853,9908.91142,9905.05432,9901.19721,9897.3401,9893.48299,9889.62588,9885.76877,9881.91166,9878.05456,9874.19745,9870.34034,…,4142.5344_diff_4138.67729,4138.67729_diff_4134.82018,4134.82018_diff_4130.96307,4130.96307_diff_4127.10596,4127.10596_diff_4123.24886,4123.24886_diff_4119.39175,4119.39175_diff_4115.53464,4115.53464_diff_4111.67753,4111.67753_diff_4107.82042,4107.82042_diff_4103.96331,4103.96331_diff_4100.10621,4100.10621_diff_4096.2491,4096.2491_diff_4092.39199,4092.39199_diff_4088.53488,4088.53488_diff_4084.67777,4084.67777_diff_4080.82066,4080.82066_diff_4076.96356,4076.96356_diff_4073.10645,4073.10645_diff_4069.24934,4069.24934_diff_4065.39223,4065.39223_diff_4061.53512,4061.53512_diff_4057.67801,4057.67801_diff_4053.82091,4053.82091_diff_4049.9638,4049.9638_diff_4046.10669,4046.10669_diff_4042.24958,4042.24958_diff_4038.39247,4038.39247_diff_4034.53536,4034.53536_diff_4030.67826,4030.67826_diff_4026.82115,4026.82115_diff_4022.96404,4022.96404_diff_4019.10693,4019.10693_diff_4015.24982,4015.24982_diff_4011.39271,4011.39271_diff_4007.5356,4007.5356_diff_4003.6785,4003.6785_diff_3999.82139
str,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""イチョウ""",1,1,216.129032,0.41485,0.41465,0.41463,0.41476,0.41481,0.4147,0.41452,0.41427,0.41392,0.41364,0.41354,0.41355,0.41352,0.41337,0.41312,0.41284,0.4127,0.41265,0.41257,0.41242,0.4123,0.41226,0.41218,0.41193,0.41159,0.41129,0.41111,0.41102,0.41096,0.41098,0.41109,0.41116,0.41104,…,0.00012,0.00119,0.0037,0.0048,0.00319,0.00056,0.00068,0.00221,0.00109,0.00074,0.00377,0.00516,0.00218,-0.00026,0.00111,0.00303,0.00464,0.0048,0.00038,-0.0022,0.00147,0.00208,-0.002,-0.00176,0.00222,0.00343,0.00312,0.00321,-0.00179,-0.0078,-0.00525,-0.00236,-0.00403,-0.00163,0.00269,0.00267,-0.00135
"""イチョウ""",2,1,210.752688,0.42049,0.4204,0.42049,0.42053,0.42038,0.4201,0.41988,0.41966,0.41942,0.41932,0.41934,0.41934,0.41929,0.41914,0.41887,0.41858,0.41838,0.41829,0.41825,0.4182,0.41809,0.41794,0.4177,0.41753,0.41749,0.41739,0.41728,0.41727,0.41723,0.4171,0.41689,0.41664,0.41651,…,0.00108,0.00158,0.00207,0.00252,0.00331,0.00332,0.00205,0.00081,0.00072,0.00158,0.00297,0.00328,0.00106,-0.00084,0.00041,0.00348,0.00503,0.00338,0.00079,0.0006,0.00463,0.00828,0.00501,-0.00085,-0.0008,0.00289,0.00169,-0.00268,-0.00318,-0.00046,0.0018,-0.00065,-0.00475,-0.00131,0.00434,0.00393,-0.00055
"""イチョウ""",3,1,205.913979,0.4104,0.41045,0.41047,0.41028,0.41,0.40989,0.40992,0.40982,0.40951,0.4092,0.40908,0.40906,0.40907,0.40913,0.40901,0.40863,0.40831,0.40809,0.40775,0.40743,0.40734,0.40736,0.40732,0.40726,0.40728,0.4072,0.40695,0.40673,0.40659,0.4065,0.40647,0.40655,0.40668,…,0.00313,0.00222,0.00151,0.0029,0.0033,0.00219,0.00227,0.00244,0.0016,0.0019,0.00368,0.00433,0.00165,-0.00081,0.00112,0.00318,0.00252,0.00219,0.00294,0.00353,0.00242,-0.00018,-0.00166,-0.00138,0.00101,0.00619,0.00968,0.00528,-0.00324,-0.00536,0.00022,0.00365,-0.00043,-0.00755,-0.00859,-0.00202,0.00264
"""イチョウ""",4,1,201.075269,0.4008,0.40061,0.40031,0.40008,0.39993,0.39991,0.39998,0.39995,0.39967,0.39931,0.39919,0.39924,0.39921,0.39905,0.39887,0.39878,0.39865,0.39837,0.39809,0.39804,0.39814,0.39808,0.39775,0.39751,0.39752,0.39749,0.3974,0.39739,0.39732,0.39707,0.39683,0.3967,0.39655,…,0.00331,0.00286,0.0016,0.00126,0.00183,0.00189,0.00145,0.00131,0.00177,0.00338,0.00383,0.00135,-0.00031,0.00151,0.00452,0.00648,0.00568,0.00087,-0.00297,-0.00074,0.00348,0.00361,-0.00068,-0.0023,0.0012,0.00238,0.0018,0.00272,0.00068,-0.00224,-0.0015,0.00052,0

In [25]:
Ginkgo = pl_diff_df_var.filter(pl.col("樹種") == "イチョウ")

In [26]:
Ginkgo.head(1)

樹種,sample number,species number,含水率,9993.76781,9989.9107,9986.05359,9982.19648,9978.33937,9974.48227,9970.62516,9966.76805,9962.91094,9959.05383,9955.19672,9951.33962,9947.48251,9943.6254,9939.76829,9935.91118,9932.05407,9928.19697,9924.33986,9920.48275,9916.62564,9912.76853,9908.91142,9905.05432,9901.19721,9897.3401,9893.48299,9889.62588,9885.76877,9881.91166,9878.05456,9874.19745,9870.34034,…,4142.5344_diff_4138.67729,4138.67729_diff_4134.82018,4134.82018_diff_4130.96307,4130.96307_diff_4127.10596,4127.10596_diff_4123.24886,4123.24886_diff_4119.39175,4119.39175_diff_4115.53464,4115.53464_diff_4111.67753,4111.67753_diff_4107.82042,4107.82042_diff_4103.96331,4103.96331_diff_4100.10621,4100.10621_diff_4096.2491,4096.2491_diff_4092.39199,4092.39199_diff_4088.53488,4088.53488_diff_4084.67777,4084.67777_diff_4080.82066,4080.82066_diff_4076.96356,4076.96356_diff_4073.10645,4073.10645_diff_4069.24934,4069.24934_diff_4065.39223,4065.39223_diff_4061.53512,4061.53512_diff_4057.67801,4057.67801_diff_4053.82091,4053.82091_diff_4049.9638,4049.9638_diff_4046.10669,4046.10669_diff_4042.24958,4042.24958_diff_4038.39247,4038.39247_diff_4034.53536,4034.53536_diff_4030.67826,4030.67826_diff_4026.82115,4026.82115_diff_4022.96404,4022.96404_diff_4019.10693,4019.10693_diff_4015.24982,4015.24982_diff_4011.39271,4011.39271_diff_4007.5356,4007.5356_diff_4003.6785,4003.6785_diff_3999.82139
str,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""イチョウ""",1,1,216.129032,0.41485,0.41465,0.41463,0.41476,0.41481,0.4147,0.41452,0.41427,0.41392,0.41364,0.41354,0.41355,0.41352,0.41337,0.41312,0.41284,0.4127,0.41265,0.41257,0.41242,0.4123,0.41226,0.41218,0.41193,0.41159,0.41129,0.41111,0.41102,0.41096,0.41098,0.41109,0.41116,0.41104,…,0.00012,0.00119,0.0037,0.0048,0.00319,0.00056,0.00068,0.00221,0.00109,0.00074,0.00377,0.00516,0.00218,-0.00026,0.00111,0.00303,0.00464,0.0048,0.00038,-0.0022,0.00147,0.00208,-0.002,-0.00176,0.00222,0.00343,0.00312,0.00321,-0.00179,-0.0078,-0.00525,-0.00236,-0.00403,-0.00163,0.00269,0.00267,-0.00135


In [32]:
import lightgbm as lgb
import polars as pl

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [33]:
# --- データ ---
pl_df = Ginkgo

# 目的変数
y = pl_df["含水率_log"].to_numpy()

# 特徴量（不要列を除外）
drop_cols = ["含水率", "含水率_log", "樹種", "sample number", "species number"]
feature_cols = [c for c in pl_df.columns if c not in drop_cols]

X = pl_df.select(feature_cols).to_numpy()
# --- 分割 ---
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- モデル ---
model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=-1,
)

# --- 学習 ---
model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="rmse",
    callbacks=[lgb.early_stopping(50)]
)

# --- 評価 ---
#pred = model.predict(X_valid)
#rmse = mean_squared_error(y_valid, pred, squared=False)

#print("RMSE:", rmse)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003303 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 74402
[LightGBM] [Info] Number of data points in the train set: 75, number of used features: 3109
[LightGBM] [Info] Start training from score 4.011418
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [35]:
pred = model.predict(X_valid)

/usr/local/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [36]:
from sklearn.metrics import mean_squared_error
import numpy as np

mse = mean_squared_error(y_valid, pred)
rmse = np.sqrt(mse)
print("RMSE:", rmse)

RMSE: 0.04213135913460014
